In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [6]:
bureau = pd.read_csv('bureau.csv')

In [2]:
df = pd.read_csv('bureau_balance.csv')

In [5]:
df.shape

(27299925, 3)

In [3]:
df.head()

,SK_ID_BUREAU,MONTHS_BALANCE,STATUS
0,5715448,0,C
1,5715448,-1,C
2,5715448,-2,C
3,5715448,-3,C
4,5715448,-4,C


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27299925 entries, 0 to 27299924
Data columns (total 3 columns):
 #   Column          Dtype 
---  ------          ----- 
 0   SK_ID_BUREAU    int64 
 1   MONTHS_BALANCE  int64 
 2   STATUS          object
dtypes: int64(2), object(1)
memory usage: 624.8+ MB


Bureau Mapping

In [7]:
bureau_mapping = (
    bureau[['SK_ID_BUREAU', 'SK_ID_CURR']]
    .drop_duplicates()
)

In [10]:
print("Rows:", len(bureau_mapping))
print("Unique bureau IDs:", bureau_mapping['SK_ID_BUREAU'].nunique())
print("Unique customers:", bureau_mapping['SK_ID_CURR'].nunique())
bureau_mapping['SK_ID_BUREAU'].duplicated().sum()

Rows: 1716428
Unique bureau IDs: 1716428
Unique customers: 305811


np.int64(0)

In [9]:
bureau_mapping.head()

,SK_ID_BUREAU,SK_ID_CURR
0,5714462,215354
1,5714463,215354
2,5714464,215354
3,5714465,215354
4,5714466,215354


Connect it to customers

In [11]:
bureau_balance = df.merge(
    bureau_mapping,
    on='SK_ID_BUREAU',
    how='left'
)

In [16]:
bureau_balance.head(5)

,SK_ID_BUREAU,MONTHS_BALANCE,STATUS,SK_ID_CURR
0,5715448,0,C,380361.0
1,5715448,-1,C,380361.0
2,5715448,-2,C,380361.0
3,5715448,-3,C,380361.0
4,5715448,-4,C,380361.0


Aggregation

In [17]:
bureau_balance['HAS_DPD'] = (
    bureau_balance['STATUS'].isin(['1','2','3','4','5'])
    .astype(int)
)

In [25]:
bb_features = (
    bureau_balance
    .groupby('SK_ID_CURR')
    .agg(
        BUREAU_BALANCE_TOTAL_RECORDS=('SK_ID_BUREAU', 'count'),
        BUREAU_BALANCE_TOTAL_DPD_MONTHS=('HAS_DPD', 'sum'),
        BUREAU_BALANCE_UNIQUE_CREDITS=('SK_ID_BUREAU', 'nunique'),
        BUREAU_BALANCE_MIN_MONTH=('MONTHS_BALANCE', 'min'),
        BUREAU_BALANCE_MAX_MONTH=('MONTHS_BALANCE', 'max')
    )
)

bb_features['BUREAU_BALANCE_DPD_RATIO'] = (
    bb_features['BUREAU_BALANCE_TOTAL_DPD_MONTHS'] /
    bb_features['BUREAU_BALANCE_TOTAL_RECORDS']
)

final_bureau_balance = (
    bb_features
    .reset_index()
)

FINAL CHECK

In [28]:
final_bureau_balance.head()

,SK_ID_CURR,BUREAU_BALANCE_TOTAL_RECORDS,BUREAU_BALANCE_TOTAL_DPD_MONTHS,BUREAU_BALANCE_UNIQUE_CREDITS,BUREAU_BALANCE_MIN_MONTH,BUREAU_BALANCE_MAX_MONTH,BUREAU_BALANCE_DPD_RATIO
0,100001.0,172,1,7,-51,0,0.005814
1,100002.0,110,27,8,-47,0,0.245455
2,100005.0,21,0,3,-12,0,0.000000
3,100010.0,72,0,2,-90,-2,0.000000
4,100013.0,230,7,4,-68,0,0.030435


In [26]:
print("Shape:", final_bureau_balance.shape)

print("Unique customers:",
      final_bureau_balance.index.nunique())

print("Duplicate customers:",
      final_bureau_balance.index.duplicated().sum())

print("\nMissing values:")
display(final_bureau_balance.isna().sum().sort_values(ascending=False))

Shape: (134542, 7)
Unique customers: 134542
Duplicate customers: 0

Missing values:


,0
SK_ID_CURR,0
BUREAU_BALANCE_TOTAL_RECORDS,0
BUREAU_BALANCE_TOTAL_DPD_MONTHS,0
BUREAU_BALANCE_UNIQUE_CREDITS,0
BUREAU_BALANCE_MIN_MONTH,0
BUREAU_BALANCE_MAX_MONTH,0
BUREAU_BALANCE_DPD_RATIO,0


**SAVE IT**

In [27]:
final_bureau_balance.to_csv(
    'bureau_balance_aggregated.csv',
    index=False
)